# Music Genre Classification Data Transformation & Preprocessing

## Objective

This notebook covers the **Data Transformation** phase, preparing everything for our PyTorch model.

Our pipeline follows three stages:

1. **EDA** — completed (class distribution, waveforms, spectrograms, MFCCs, 
   corrupted/duplicate file detection)
2. **SQL schema & cleaning** — completed (metadata table built from EDA findings: 
   duplicate/corrupted flags, train/val/test split assigned at track level)
3. **Data Transformation** — this notebook (preprocessing pipeline: trim/pad, 
   windowing, MFCC extraction, normalization, augmentation)

## Goal
Create a PyTorch Dataset/DataLoader from clean SQL metadata we expect by the end of this notebook, we will have a working PyTorch `Dataset` and `DataLoader`
### Workflow — preprocess_pytorch.ipynb

- **Set random seed (42)** — numpy, torch, and Python's `random` module, for reproducible splits and results
- **Connect to SQL** — query `vw_clean_tracks` to pull `file_path`, `label`, `genre_id`, `split` for all 971 clean tracks
- **Trim/pad audio** — standardize every clip to exactly 30 seconds, save to `processed/`
- **GTZANDataset class** — for each track requested:
  - slice a 3-second window (random for train, fixed/centered for val/test)
  - apply augmentation — time shift, noise, frequency masking (train only)
  - compute the full 2D MFCC array (n_mfcc=20)
  - normalize using train-only mean/std
  - add channel dimension for Conv2d input
- **Compute normalization stats** — mean/std from train split only, reused across all splits
- **Recreate train/val/test datasets** — with normalization applied
- **Wrap in DataLoader** — verify batch shape and value range before moving to model development

In [9]:
# --- Standard library ---
import os
import random
from pathlib import Path

# --- Data & numeric ---
import numpy as np
import pandas as pd

# --- Audio processing ---
import librosa
import soundfile as sf

# --- Database connection ---
import psycopg2
from dotenv import load_dotenv

# --- PyTorch ---
import torch
from torch.utils.data import Dataset, DataLoader

# --- SQLAlchemy ---
from sqlalchemy import create_engine


# --- PyTorch modules ---
import torch.nn.functional as F

In [2]:
## Set random seeds for reproducibility 
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Conexión SQL (971 clean_tracks)

In [3]:

# load environment variables from .env file 
load_dotenv()

# read DB connection details from environment variables
DB_NAME = os.getenv("DB_NAME", "music_genre_db")  # database name, default to music_genre_db
DB_USER = os.getenv("DB_USER", "ingxrodriguez")    # postgres user
DB_PASSWORD = os.getenv("DB_PASSWORD")              # postgres password, must come from .env, never hardcoded
DB_HOST = os.getenv("DB_HOST", "localhost")         # db host
DB_PORT = os.getenv("DB_PORT", "5432")              # db port

# create a SQLAlchemy engine using those credentials
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# query the clean tracks view: excludes corrupted_flagged and duplicate_flagged rows
query = "SELECT track_id, file_path, label, genre_id, split FROM vw_clean_tracks;"

# read the query result directly into a pandas DataFrame, using the engine 
clean_tracks_df = pd.read_sql(query, engine)

# quick sanity check: confirm row count and preview the first few rows
print("Total clean tracks:", len(clean_tracks_df))
clean_tracks_df.head()

Total clean tracks: 971


,track_id,file_path,label,genre_id,split
0,16,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
1,41,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
2,3,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
3,78,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val
4,28,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val


# Trim/pad a 30 seg

### Why we trim/pad to 30 seconds

Our EDA showed that GTZAN clips are almost all ~30 seconds long, but not 
perfectly uniform a few are slightly shorter or longer. Since our model 
extracts MFCCs as full 2D arrays (coefficients × time), every clip must 
have the exact same number of audio samples; otherwise the resulting MFCC 
arrays will have different time dimensions and can't be batched together 
by the DataLoader.

To fix this, we standardize every clip to exactly 30 seconds (22050 Hz × 30 
= 661,500 samples):
- Clips longer than 30s are **trimmed** down to the target length.
- Clips shorter than 30s are **padded** with silence (zeros) at the end.

Original files are never modified  trimmed/padded copies are saved to a 
separate `processed/` folder, preserving the raw dataset as our permanent 
reference.

In [4]:
# native sample rate confirmed identical across all readable files during EDA (22050 Hz)
TARGET_SR = 22050

# standard clip length decided during cleaning: 30 seconds (matches original GTZAN clip length)
TARGET_DURATION_SEC = 30

# convert target duration to number of samples (30 sec * 22050 samples/sec)
TARGET_LENGTH_SAMPLES = TARGET_SR * TARGET_DURATION_SEC

# root folder where original, untouched audio files live
DATA_ROOT = Path("../Data_Music")

# new folder for trimmed/padded copies -- originals are never overwritten
PROCESSED_DIR = DATA_ROOT / "processed"


def trim_or_pad(y, target_length):
    """Trim audio array to target_length, or pad with zeros (silence) if shorter."""
    current_length = len(y)  # number of samples in the loaded audio

    if current_length > target_length:
        # clip is longer than target: cut it down to exactly target_length samples
        return y[:target_length]
    elif current_length < target_length:
        # clip is shorter than target: pad the end with zeros (silence) up to target_length
        pad_amount = target_length - current_length
        return np.pad(y, (0, pad_amount), mode="constant")
    else:
        # already exactly the target length, no change needed
        return y


# will store the new, trimmed file path for each track so we can save it back to the dataframe
processed_paths = []

# loop through every clean track from our SQL query
for row in clean_tracks_df.itertuples(index=False):
    # load the original audio at its verified native sample rate (sr=None reads actual rate, doesn't impose one)
    y, sr = librosa.load(row.file_path, sr=None)

    # apply trim/pad so every clip has exactly TARGET_LENGTH_SAMPLES samples
    y_fixed = trim_or_pad(y, TARGET_LENGTH_SAMPLES)

    # build the output subfolder path, mirroring genre structure: processed/<label>/
    genre_subdir = PROCESSED_DIR / row.label
    genre_subdir.mkdir(parents=True, exist_ok=True)  # create folder if it doesn't exist yet

    # build the output file path using the same filename as the original
    out_path = genre_subdir / Path(row.file_path).name

    # write the trimmed/padded audio to disk (original file is untouched)
    sf.write(out_path, y_fixed, sr)

    # record the new path so we can update our dataframe afterward
    processed_paths.append(str(out_path))

# add the processed file path as a new column, keeping the original file_path column intact
clean_tracks_df["processed_path"] = processed_paths

# confirm how many files were processed
print(f"Trimmed/padded {len(clean_tracks_df)} files to {TARGET_DURATION_SEC} sec each.")
clean_tracks_df.head()

Trimmed/padded 971 files to 30 sec each.


,track_id,file_path,label,genre_id,split,processed_path
0,16,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00015.wav
1,41,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00040.wav
2,3,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00002.wav
3,78,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val,../Data_Music/processed/blues/blues.00077.wav
4,28,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val,../Data_Music/processed/blues/blues.00027.wav


### Summary: GTZANDataset Class

Wraps the full feature-extraction pipeline into a single PyTorch `Dataset`, 
producing a ResNet18-ready input for each track:

- Filters to one split ("train", "val", or "test")
- Slices a 3-second window from the 30-sec audio (random for train, fixed/centered for val/test)
- Applies augmentation — time shift, noise, frequency masking (train only)
- Computes the full 2D MFCC array (20 coefficients × time, not averaged)
- Normalizes using train-only mean/std
- Resizes the MFCC from its native (20, T) shape up to (128, 128), so ResNet18's 
  downsampling stages have enough spatial structure to work with
- Duplicates the single channel 3 times → (3, 128, 128), since pretrained 
  ImageNet models expect 3-channel (RGB) input






In [10]:
class GTZANDataset(Dataset):
    """PyTorch Dataset that loads a 3-second window of audio and returns a normalized,
    ResNet18-ready MFCC "image" + genre label.
    Train uses a random window each call (dynamic variety); val/test use a fixed centered window (reproducible eval).
    Applies data augmentation (time shift, noise, frequency masking) ONLY when split == 'train'."""

    def __init__(self, metadata_df, split, n_mfcc=20, mfcc_mean=None, mfcc_std=None,
                 window_duration_sec=3, sample_rate=22050, resize_to=(128, 128)):
        self.df = metadata_df[metadata_df["split"] == split].reset_index(drop=True)
        self.split = split
        self.n_mfcc = n_mfcc
        self.mfcc_mean = mfcc_mean
        self.mfcc_std = mfcc_std
        self.window_length_samples = int(window_duration_sec * sample_rate)

        # target (height, width) for resizing the MFCC before feeding it to ResNet18. ResNet18 was designed
        # for ~224x224 images; our native MFCC (20, 130) is too small for its 5 downsampling stages to keep
        # any meaningful spatial structure by the final layer, so we resize up to give the network enough
        # room to use its full depth
        self.resize_to = resize_to

    def __len__(self):
        return len(self.df)

    def _get_window(self, y):
        max_start = len(y) - self.window_length_samples
        if self.split == "train":
            start = random.randint(0, max_start)
        else:
            start = max_start // 2
        return y[start:start + self.window_length_samples]

    def _time_shift(self, y):
        max_shift = int(0.1 * len(y))
        shift_amount = random.randint(-max_shift, max_shift)
        return np.roll(y, shift_amount)

    def _add_noise(self, y, noise_level=0.005):
        noise = np.random.randn(len(y)) * noise_level
        return y + noise

    def _frequency_mask(self, mfcc, max_mask_width=4):
        n_mfcc, T = mfcc.shape
        mask_width = random.randint(1, max_mask_width)
        start = random.randint(0, n_mfcc - mask_width)
        mfcc_masked = mfcc.copy()
        mfcc_masked[start:start + mask_width, :] = 0.0
        return mfcc_masked

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y, sr = librosa.load(row["processed_path"], sr=None)
        y = self._get_window(y)

        if self.split == "train":
            y = self._time_shift(y)
            y = self._add_noise(y)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.n_mfcc)

        if self.split == "train":
            mfcc = self._frequency_mask(mfcc)

        mfcc_tensor = torch.tensor(mfcc, dtype=torch.float32)

        if self.mfcc_mean is not None and self.mfcc_std is not None:
            mfcc_tensor = (mfcc_tensor - self.mfcc_mean) / self.mfcc_std

        # --- Adapting our MFCC "image" for a pretrained torchvision CNN (ResNet18) ---

        # Step 1: add a channel dimension so F.interpolate sees it as (batch=1, channel=1, H, W)
        mfcc_tensor = mfcc_tensor.unsqueeze(0).unsqueeze(0)  # (n_mfcc, T) -> (1, 1, n_mfcc, T)

        # Step 2: resize from our native (20, T) shape up to (128, 128), so ResNet18's downsampling
        # stages have enough spatial structure left to work with by the final layer
        mfcc_tensor = F.interpolate(mfcc_tensor, size=self.resize_to, mode="bilinear", align_corners=False)

        # Step 3: drop the temporary batch dim added in Step 1, leaving (1, 128, 128)
        mfcc_tensor = mfcc_tensor.squeeze(0)

        # Step 4: pretrained ImageNet models expect 3 input channels (RGB); duplicate our single
        # channel 3 times rather than modifying ResNet18's first layer, so ALL pretrained weights stay intact
        mfcc_tensor = mfcc_tensor.repeat(3, 1, 1)  # (1, 128, 128) -> (3, 128, 128)

        label = torch.tensor(row["genre_id"], dtype=torch.long)

        return mfcc_tensor, label

### Compute normalization stats (train split only)

Creates a temporary `GTZANDataset` for the train split with no `mfcc_mean`/`mfcc_std` 
passed in, so it returns raw, unnormalized MFCCs. Loops through every training 
track, collects its raw MFCC tensor, and stacks them all into one tensor to compute 
a single global mean and standard deviation.

These two numbers (`MFCC_MEAN`, `MFCC_STD`) are computed **only** from train never 
from val or test and will be reused to normalize all three splits. This prevents 
data leakage: no information about the validation or test sets influences how the 
data gets normalized.

In [11]:
# calculate the mean and std of the MFCCs from the train split only, to be used for normalization in all splits
# create a temporary train-only dataset with no mean/std passed in, so MFCCs come back raw (not normalized yet)
train_dataset_raw = GTZANDataset(clean_tracks_df, split="train")

# empty list to collect every raw MFCC tensor from the train split
all_train_mfccs = []
# loop through every track in the train split
for i in range(len(train_dataset_raw)):
# get the raw MFCC tensor for this track, ignore the label (we only need MFCC values here)
    mfcc_tensor, _ = train_dataset_raw[i]
# add this track's MFCC tensor to our list
    all_train_mfccs.append(mfcc_tensor)
# stack the list of individual MFCC tensors into one big tensor: shape (num_train_tracks, 20, T)
stacked_train_mfccs = torch.stack(all_train_mfccs)
# compute a single global mean across all train MFCC values (all tracks, all coefficients, all time frames)
MFCC_MEAN = stacked_train_mfccs.mean()
# compute a single global standard deviation across all train MFCC values
MFCC_STD = stacked_train_mfccs.std()
# print the computed stats so we can document and reproduce them later
print("Train MFCC mean:", MFCC_MEAN.item())
print("Train MFCC std:", MFCC_STD.item())

Train MFCC mean: -0.6547410488128662
Train MFCC std: 32.05534744262695


### Recreate datasets with normalization applied

Creates the three final `GTZANDataset` objects — one per split this time passing 
in `MFCC_MEAN` and `MFCC_STD` (computed from train only), so every MFCC returned by 
`__getitem__` is now normalized.

Then runs a quick sanity check: pulls the first training sample to confirm its 
shape/label look right, prints the size of each split (expected: 677 / 141 / 153), 
and prints the normalized sample's mean/std — these should land close to 0 and 1 
respectively, confirming the normalization was applied correctly.

In [12]:
# calling our class GTZANDataset to create 3 objects/instances for train, val, and test splits, 
# passing in the mean and std computed from train only, so all 3 splits normalize the same way
train_dataset = GTZANDataset(clean_tracks_df, split="train", mfcc_mean=MFCC_MEAN, mfcc_std=MFCC_STD)
val_dataset = GTZANDataset(clean_tracks_df, split="val", mfcc_mean=MFCC_MEAN, mfcc_std=MFCC_STD)
test_dataset = GTZANDataset(clean_tracks_df, split="test", mfcc_mean=MFCC_MEAN, mfcc_std=MFCC_STD)

# test that the datasets work correctly by retrieving the first sample from train and checking its shape/label
sample_mfcc, sample_label = train_dataset[0]
# confirm how many tracks landed in each split, and confirm normalization worked (mean near 0, std near 1)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))
print("Normalized sample MFCC mean:", sample_mfcc.mean().item())
print("Normalized sample MFCC std:", sample_mfcc.std().item())

Train size: 677
Val size: 141
Test size: 153
Normalized sample MFCC mean: -0.07121793180704117
Normalized sample MFCC std: 1.2870006561279297


### Wrap datasets in a DataLoader and verify a batch

Wraps each `GTZANDataset` in a `DataLoader`. Train shuffles tracks every epoch 
(`shuffle=True`) so the model sees genres in a different order each time; val/test 
don't need shuffling since they're only used for evaluation, not training.

Pulls one batch from `train_loader` as a final end-to-end check before moving to 
model development — confirming:
- **Batch MFCC shape**: `(32, 3, 128, 128)` — batch size, 3 duplicated channels 
  (for pretrained ResNet18), resized MFCC height × width (128×128)
- **Batch labels shape**: `(32,)` — one genre label per track
- **Value range**: should be centered near 0, consistent with our normalization

**Verified output**:
- `Batch MFCC shape: torch.Size([32, 3, 128, 128])`
- `Batch labels shape: torch.Size([32])`
- `Batch MFCC value range: min = -9.14, max = 4.55`

In [13]:
# batch size: how many tracks are grouped together per training step
BATCH_SIZE = 32

# train loader: shuffle=True so the model sees genres in a different random order each epoch
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# val/test loaders: shuffle=False -- no need to shuffle since we're only evaluating, not training
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# pull one batch from the train loader to verify everything works end-to-end
batch_mfcc, batch_labels = next(iter(train_loader))

# expected shape: (BATCH_SIZE, n_mfcc, T) -- e.g. (32, 20, 1292)
print("Batch MFCC shape:", batch_mfcc.shape)

# expected shape: (BATCH_SIZE,) -- one genre_id per track in the batch
print("Batch labels shape:", batch_labels.shape)

# sanity check on value range -- should be centered near 0 given our normalization
print("Batch MFCC value range: min =", batch_mfcc.min().item(), " max =", batch_mfcc.max().item())

Batch MFCC shape: torch.Size([32, 3, 128, 128])
Batch labels shape: torch.Size([32])
Batch MFCC value range: min = -9.143953323364258  max = 4.550596714019775


# Conclusion
### Conclusion

The preprocessing pipeline is fully verified end-to-end: metadata comes from a 
clean, deduplicated, leakage-free SQL split (677/141/153 tracks); each track is 
sliced into a 3-second window (random for train, fixed/centered for val/test), 
augmented (train only), and converted into a full 2D MFCC array (20 coefficients, 
not averaged). The MFCC is then resized from its native (20, T) shape up to 
(128, 128) and its single channel duplicated 3 times, producing a `(3, 128, 128)` 
tensor compatible with a pretrained ResNet18 — while keeping all of ResNet18's 
pretrained weights intact.

Each batch returns `torch.Size([32, 3, 128, 128])` for the MFCC tensors and 
`torch.Size([32])` for the labels, with values centered near 0 (range roughly 
-9.14 to 4.55), confirming normalization was applied correctly. Data augmentation 
is applied only to the training split, and the train/val/test split is assigned 
at the original-track level in SQL, before any windowing occurs — so no audio 
from a given song can appear in more than one split.

The pipeline is ready to feed directly into a pretrained ResNet18 for Sprint 3 
model development.